In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.payer_master_patient_level WHERE INSURANCE_GROUP = 'Unknown';

In [0]:
SELECT COUNT(DISTINCT PATIENT_ID) FROM  com_edp_prd.cmpa_insights_internal_schema.patient360_master;

In [0]:
SELECT 
    INSURANCE_GROUP,
    AGE_GROUP,

    COUNT(DISTINCT CLAIM_ID) AS CLAIM_COUNT,
    COUNT(DISTINCT CASE WHEN ELAPRASE_FLAG = 1 THEN CLAIM_ID END) AS ELAPRASE_CLAIM_COUNT

FROM
(

SELECT DISTINCT 
    C.CLAIM_ID,
    P360.PATIENT_ID,

    YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) AS CURRENT_AGE,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) < 18 THEN '<18'
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) >= 18 THEN '>=18'
    END AS AGE_GROUP,

    PL.INSURANCE_GROUP,

    CASE 
        WHEN C.CODE IN ('54092070001','540920700','J1743') THEN 1
        ELSE 0
    END AS ELAPRASE_FLAG

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master P360

LEFT JOIN
(
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY PATIENT_ID
) D
ON P360.PATIENT_ID = D.PATIENT_ID

LEFT JOIN
(

    /* Medical claims */
    SELECT
        PATIENT_ID,
        MEDICAL_EVENT_ID AS CLAIM_ID,
        SERVICE_DATE AS CLAIM_DATE,
        KH_PLAN_ID AS PLAN_ID,
        NDC11 AS CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

    UNION ALL

    SELECT
        PATIENT_ID,
        MEDICAL_EVENT_ID,
        SERVICE_DATE,
        KH_PLAN_ID,
        PROCEDURE_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

    UNION ALL

    /* Pharmacy claims */
    SELECT
        PATIENT_ID,
        PHARMACY_EVENT_ID,
        FILL_DATE,
        COALESCE(PRIMARY_KH_PLAN_ID,SECONDARY_KH_PLAN_ID),
        NDC11
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
      AND TRANSACTION_RESULT = 'PAID'

) C
ON P360.PATIENT_ID = C.PATIENT_ID

LEFT JOIN com_edp_prd.com_raw.kom_plans PL
ON C.PLAN_ID = PL.KH_PLAN_ID

) BASE

GROUP BY 
    INSURANCE_GROUP,
    AGE_GROUP
ORDER BY 
    AGE_GROUP,
    INSURANCE_GROUP;

In [0]:
SELECT 
    INSURANCE_GROUP,
    AGE_GROUP,

    COUNT(DISTINCT CLAIM_ID) AS CLAIM_COUNT,
    COUNT(DISTINCT CASE WHEN ELAPRASE_FLAG = 1 THEN CLAIM_ID END) AS ELAPRASE_CLAIM_COUNT

FROM
(

SELECT DISTINCT 
    C.CLAIM_ID,
    P360.PATIENT_ID,

    YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) AS CURRENT_AGE,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) < 18 THEN '<18'
        ELSE '>=18'
    END AS AGE_GROUP,

    /* Insurance group mapping */
    CASE 
        WHEN PL.INSURANCE_GROUP ILIKE '%copay card%' THEN 'Commercial'
        WHEN PL.INSURANCE_GROUP ILIKE '%discount card%' THEN 'Others'
        WHEN PL.INSURANCE_GROUP ILIKE '%not insurance%' THEN 'Others'
        WHEN PL.INSURANCE_GROUP ILIKE '%special program%' THEN 'Others'
        WHEN PL.INSURANCE_GROUP ILIKE '%workers compensation%' THEN 'Others'
        WHEN PL.INSURANCE_GROUP ILIKE 'va%' THEN 'Others'
        WHEN PL.INSURANCE_GROUP ILIKE '%assistance%' THEN 'Others'

        WHEN PL.INSURANCE_GROUP ILIKE '%hospice%' THEN 'Medicare'
        WHEN PL.INSURANCE_GROUP ILIKE '%spap%' THEN 'Medicare'
        WHEN PL.INSURANCE_GROUP ILIKE '%medicare dme%' THEN 'Medicare'

        WHEN PL.INSURANCE_GROUP ILIKE '%commercial%' THEN 'Commercial'
        WHEN PL.INSURANCE_GROUP ILIKE '%medicaid%' THEN 'Medicaid'
        WHEN PL.INSURANCE_GROUP ILIKE '%medicare%' THEN 'Medicare'

        ELSE 'Others'
    END AS INSURANCE_GROUP,

    CASE 
        WHEN C.CODE IN ('54092070001','540920700','J1743') THEN 1
        ELSE 0
    END AS ELAPRASE_FLAG

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master P360

LEFT JOIN
(
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY PATIENT_ID
) D
ON P360.PATIENT_ID = D.PATIENT_ID

LEFT JOIN
(

    /* Medical claims - NDC */
    SELECT
        PATIENT_ID,
        MEDICAL_EVENT_ID AS CLAIM_ID,
        SERVICE_DATE AS CLAIM_DATE,
        KH_PLAN_ID AS PLAN_ID,
        NDC11 AS CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

    UNION ALL

    /* Medical claims - procedure */
    SELECT
        PATIENT_ID,
        MEDICAL_EVENT_ID,
        SERVICE_DATE,
        KH_PLAN_ID,
        PROCEDURE_CODE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

    UNION ALL

    /* Pharmacy claims */
    SELECT
        PATIENT_ID,
        PHARMACY_EVENT_ID,
        FILL_DATE,
        COALESCE(PRIMARY_KH_PLAN_ID,SECONDARY_KH_PLAN_ID),
        NDC11
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
      AND TRANSACTION_RESULT = 'PAID'

) C
ON P360.PATIENT_ID = C.PATIENT_ID

LEFT JOIN com_edp_prd.com_raw.kom_plans PL
ON C.PLAN_ID = PL.KH_PLAN_ID

) BASE

GROUP BY 
    INSURANCE_GROUP,
    AGE_GROUP

ORDER BY 
    AGE_GROUP,
    INSURANCE_GROUP;

In [0]:
SELECT DISTINCT
    P360.PATIENT_ID,

    YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) AS AGE,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) < 18 THEN '<18'
        ELSE '>=18'
    END AS AGE_GROUP,

    COUNT(DISTINCT C.CLAIM_ID) AS CLAIM_COUNT,

    C.PLAN_ID AS CLAIM_PLAN_ID,
    PL.PAYER_ID AS CLAIM_PAYER_ID,
    PL.PAYER_NAME AS CLAIM_PAYER_NAME,
    PL.INSURANCE_GROUP AS CLAIM_INSURANCE_GROUP

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master P360

LEFT JOIN
(
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY PATIENT_ID
) D
ON P360.PATIENT_ID = D.PATIENT_ID


/* ================= CLAIMS ================= */

LEFT JOIN
(
    SELECT
        PATIENT_ID,
        MEDICAL_EVENT_ID AS CLAIM_ID,
        KH_PLAN_ID AS PLAN_ID
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

    UNION

    SELECT
        PATIENT_ID,
        PHARMACY_EVENT_ID AS CLAIM_ID,
        COALESCE(PRIMARY_KH_PLAN_ID,SECONDARY_KH_PLAN_ID) AS PLAN_ID
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
      AND TRANSACTION_RESULT = 'PAID'

) C
ON P360.PATIENT_ID = C.PATIENT_ID


LEFT JOIN com_edp_prd.com_raw.kom_plans PL
ON C.PLAN_ID = PL.KH_PLAN_ID


/* Filter ONLY Others */

WHERE COALESCE(PL.INSURANCE_GROUP,'') NOT IN ('COMMERCIAL','MEDICAID','MEDICARE')

GROUP BY
    P360.PATIENT_ID,
    AGE,
    AGE_GROUP,
    C.PLAN_ID,
    PL.PAYER_ID,
    PL.PAYER_NAME,
    PL.INSURANCE_GROUP;

In [0]:
SELECT 
    P360.PATIENT_ID,

    YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) AS AGE,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) < 18 THEN '<18'
        ELSE '>=18'
    END AS AGE_GROUP,

    COUNT(DISTINCT E.CLAIM_ID) AS ELAPRASE_CLAIM_COUNT,

    E.PLAN_ID AS ELAPRASE_PLAN_ID,
    EPL.PAYER_ID AS ELAPRASE_PAYER_ID,
    EPL.PAYER_NAME AS ELAPRASE_PAYER_NAME,
    EPL.INSURANCE_GROUP AS ELAPRASE_INSURANCE_GROUP

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master P360

LEFT JOIN
(
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY PATIENT_ID
) D
ON P360.PATIENT_ID = D.PATIENT_ID


/* ================= ELAPRASE CLAIMS ================= */

LEFT JOIN
(
    SELECT
        PATIENT_ID,
        CLAIM_ID,
        PLAN_ID
    FROM
    (
        SELECT
            PATIENT_ID,
            MEDICAL_EVENT_ID AS CLAIM_ID,
            KH_PLAN_ID AS PLAN_ID,
            NDC11 AS CODE
        FROM com_edp_prd.com_raw.kom_medical_events

        UNION

        SELECT
            PATIENT_ID,
            MEDICAL_EVENT_ID AS CLAIM_ID,
            KH_PLAN_ID AS PLAN_ID,
            PROCEDURE_CODE AS CODE
        FROM com_edp_prd.com_raw.kom_medical_events

        UNION

        SELECT
            PATIENT_ID,
            PHARMACY_EVENT_ID AS CLAIM_ID,
            COALESCE(PRIMARY_KH_PLAN_ID,SECONDARY_KH_PLAN_ID) AS PLAN_ID,
            NDC11 AS CODE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
    ) X
    WHERE CODE IN ('54092070001','540920700','J1743')
    and TRANSACTION_STATUS = 'PAID'

) E
ON P360.PATIENT_ID = E.PATIENT_ID


LEFT JOIN com_edp_prd.com_raw.kom_plans EPL
ON E.PLAN_ID = EPL.KH_PLAN_ID


/* Filter ONLY Others */

WHERE COALESCE(EPL.INSURANCE_GROUP,'') NOT IN ('COMMERCIAL','MEDICAID','MEDICARE')

GROUP BY
    P360.PATIENT_ID,
    AGE,
    AGE_GROUP,
    E.PLAN_ID,
    EPL.PAYER_ID,
    EPL.PAYER_NAME,
    EPL.INSURANCE_GROUP;

In [0]:
SELECT 
    EPL.INSURANCE_GROUP,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) < 18 THEN '<18'
        ELSE '>=18'
    END AS AGE_GROUP,

    COUNT(DISTINCT E.CLAIM_ID) AS ELAPRASE_CLAIM_COUNT

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master P360

LEFT JOIN
(
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY PATIENT_ID
) D
ON P360.PATIENT_ID = D.PATIENT_ID


/* Elaprase Claims */

LEFT JOIN
(
    SELECT
        PATIENT_ID,
        CLAIM_ID,
        PLAN_ID
    FROM
    (
        SELECT
            PATIENT_ID,
            MEDICAL_EVENT_ID AS CLAIM_ID,
            KH_PLAN_ID AS PLAN_ID,
            NDC11 AS CODE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

        UNION

        SELECT
            PATIENT_ID,
            MEDICAL_EVENT_ID AS CLAIM_ID,
            KH_PLAN_ID AS PLAN_ID,
            PROCEDURE_CODE AS CODE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

        UNION

        SELECT
            PATIENT_ID,
            PHARMACY_EVENT_ID AS CLAIM_ID,
            COALESCE(PRIMARY_KH_PLAN_ID,SECONDARY_KH_PLAN_ID) AS PLAN_ID,
            NDC11 AS CODE
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        where TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) X
    WHERE CODE IN ('54092070001','540920700','J1743')
) E
ON P360.PATIENT_ID = E.PATIENT_ID

LEFT JOIN com_edp_prd.com_raw.kom_plans EPL
ON E.PLAN_ID = EPL.KH_PLAN_ID


GROUP BY 
    INSURANCE_GROUP,
    AGE_GROUP

ORDER BY 
    AGE_GROUP,
    INSURANCE_GROUP;

In [0]:
SELECT DISTINCT
    P360.PATIENT_ID,

    YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) AS AGE,

    CASE 
        WHEN YEAR(CURRENT_DATE) - YEAR(D.PATIENT_YOB) < 18 THEN '<18'
        ELSE '>=18'
    END AS AGE_GROUP,

    COUNT(DISTINCT E.CLAIM_ID) AS ELAPRASE_CLAIM_COUNT,
    E.PLAN_ID AS ELAPRASE_PLAN_ID,
    EPL.PAYER_ID AS ELAPRASE_PAYER_ID,
    EPL.PAYER_NAME AS ELAPRASE_PAYER_NAME,
    EPL.INSURANCE_GROUP AS ELAPRASE_INSURANCE_GROUP

FROM com_edp_prd.cmpa_insights_internal_schema.patient360_master P360

LEFT JOIN
(
    SELECT PATIENT_ID, MAX(PATIENT_YOB) AS PATIENT_YOB
    FROM com_edp_prd.com_raw.kom_patient_demographics
    GROUP BY PATIENT_ID
) D
ON P360.PATIENT_ID = D.PATIENT_ID


/* Elaprase Claims */

LEFT JOIN
(
    SELECT
        PATIENT_ID,
        CLAIM_ID,
        PLAN_ID
    FROM
    (
        SELECT
            PATIENT_ID,
            MEDICAL_EVENT_ID AS CLAIM_ID,
            KH_PLAN_ID AS PLAN_ID,
            NDC11 AS CODE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

        UNION

        SELECT
            PATIENT_ID,
            MEDICAL_EVENT_ID,
            KH_PLAN_ID,
            PROCEDURE_CODE
        FROM com_edp_prd.com_raw.kom_medical_events
        WHERE SERVICE_DATE BETWEEN '2023-08-01' AND '2025-12-31'

        UNION

        SELECT
            PATIENT_ID,
            PHARMACY_EVENT_ID,
            COALESCE(PRIMARY_KH_PLAN_ID,SECONDARY_KH_PLAN_ID),
            NDC11
        FROM com_edp_prd.com_raw.kom_pharmacy_events
        WHERE TRANSACTION_RESULT = 'PAID'
        AND FILL_DATE BETWEEN '2023-08-01' AND '2025-12-31'
    ) X
    WHERE CODE IN ('54092070001','540920700','J1743')
) E
ON P360.PATIENT_ID = E.PATIENT_ID

LEFT JOIN com_edp_prd.com_raw.kom_plans EPL
ON E.PLAN_ID = EPL.KH_PLAN_ID


/* Filter only Others based on your mapping */

WHERE COALESCE(EPL.INSURANCE_GROUP,'') NOT IN ('COMMERCIAL','MEDICAID','MEDICARE')

GROUP BY
    P360.PATIENT_ID,
    AGE,
    AGE_GROUP,
    E.PLAN_ID,
    EPL.PAYER_ID,
    EPL.PAYER_NAME,
    EPL.INSURANCE_GROUP

ORDER BY 
    AGE_GROUP,
    P360.PATIENT_ID;